In [1]:
from typing import List, Dict


class ConversationHistory:
    def __init__(self):
        self.messages: List[Dict[str, str]] = []

    def append(self, role: str, content: str):
        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, last_n_turns: int = 5) -> List[Dict[str, str]]:
        """
        Returns the latest N conversation turns.
        One turn = user message + assistant response.
        """

        if not self.messages:
            return []

        max_messages = last_n_turns * 2

        return self.messages[-max_messages:]

    def clear(self):
        self.messages.clear()

    def replace_old_messages(self, summary: str):
        """
        Replace old conversation with a summary block.
        """
        self.messages = [
            {
                "role": "system",
                "content": f"Previous conversation summary: {summary}"
            }
        ]

    def count_turns(self) -> int:
        return len([
            message
            for message in self.messages
            if message["role"] == "user"
        ])

In [6]:
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"  # Replace with your actual OpenAI API key

In [7]:
import os
from typing import Optional

from fastapi import FastAPI
from pydantic import BaseModel

from openai import OpenAI

# from memory import ConversationHistory # This line is removed as ConversationHistory is defined in another cell


app = FastAPI(
    title="Day 24 - Conversation Memory API",
    version="1.0"
)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Session-wise memory
sessions = {}


MODEL = "gpt-4o-mini"


class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = "default"


class ChatResponse(BaseModel):
    session_id: str
    response: str
    turns: int


def get_session(session_id: str) -> ConversationHistory:

    if session_id not in sessions:
        sessions[session_id] = ConversationHistory()

    return sessions[session_id]


def format_context(history: ConversationHistory):

    context = history.get_context(last_n_turns=5)

    return context


def summarize_old_messages(history: ConversationHistory):

    if len(history.messages) <= 20:
        return

    # Oldest 5 turns = first 10 messages
    old_messages = history.messages[:10]

    conversation_text = "\n".join(
        f"{msg['role']}: {msg['content']}"
        for msg in old_messages
    )

    summary_prompt = f"""
Summarize the following conversation.

Keep:
- Important facts
- User goals
- Decisions
- Technical details
- Preferences
- Important context needed for future questions

Conversation:

{conversation_text}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You summarize conversations accurately and concisely."
            },
            {
                "role": "user",
                "content": summary_prompt
            }
        ],
        temperature=0
    )

    summary = response.choices[0].message.content

    # Keep the summary + remaining messages
    remaining_messages = history.messages[10:]

    history.messages = [
        {
            "role": "system",
            "content": f"Previous conversation summary: {summary}"
        }
    ] + remaining_messages


@app.get("/")
def home():

    return {
        "message": "Day 24 Conversation Memory API is running"
    }


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):

    session_id = request.session_id

    history = get_session(session_id)

    # Add user message
    history.append(
        "user",
        request.message
    )

    # Get last 5 turns
    context = format_context(history)

    system_message = """
You are a helpful AI assistant.

Use the conversation context to understand
follow-up questions and pronouns such as:

it, this, that, they, them, he, she.

Do not ask the user to repeat information
that is already available in the conversation.
"""

    messages = [
        {
            "role": "system",
            "content": system_message
        }
    ]

    messages.extend(context)

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7
    )

    answer = response.choices[0].message.content

    # Store assistant response
    history.append(
        "assistant",
        answer
    )

    # Summarize after exceeding 10 turns
    summarize_old_messages(history)

    return ChatResponse(
        session_id=session_id,
        response=answer,
        turns=history.count_turns()
    )


@app.delete("/session/{session_id}")
def clear_session(session_id: str):

    if session_id in sessions:
        sessions[session_id].clear()

    return {
        "message": f"Session {session_id} cleared"
    }

In [9]:
!pip install fastapi uvicorn openai pydantic python-dotenv

In [11]:
OPENAI_API_KEY='your_openai_api_key_here'

In [12]:
from dotenv import load_dotenv

load_dotenv()

False

In [13]:
import os

from dotenv import load_dotenv

load_dotenv()

from fastapi import FastAPI

In [14]:
pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [16]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [8525] using WatchFiles
ERROR:    Error loading ASGI app. Could not import module "main".
INFO:     Stopping reloader process [8525]


In [17]:

{
  "session_id": "test123",
  "message": "I am building a resume screening system."
}

{'session_id': 'test123',
 'message': 'I am building a resume screening system.'}

In [18]:
{
  "session_id": "test123",
  "message": "It uses TF-IDF and machine learning."
}

{'session_id': 'test123', 'message': 'It uses TF-IDF and machine learning.'}

In [19]:

{
  "session_id": "test123",
  "message": "How can I improve it?"
}

{'session_id': 'test123', 'message': 'How can I improve it?'}

In [20]:

{
  "session_id": "test123",
  "message": "I also want to add an ATS score."
}

{'session_id': 'test123', 'message': 'I also want to add an ATS score.'}

In [21]:
{
  "session_id": "test123",
  "message": "Would that make it more useful?"
}

{'session_id': 'test123', 'message': 'Would that make it more useful?'}